In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
Raw_Data_Path = Path(r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\01_Raw_Data")
Clean_Data_Path = Path(r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\02_Clean_Data")

print("Raw data folder:", Raw_Data_Path.resolve())
print("Clean data folder:", Clean_Data_Path.resolve())

Raw data folder: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\01_Raw_Data
Clean data folder: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\02_Clean_Data


In [3]:
customers = pd.read_csv(Raw_Data_Path / r"olist_customers_dataset.csv")
geolocation = pd.read_csv(Raw_Data_Path / r"olist_geolocation_dataset.csv")
order_items = pd.read_csv(Raw_Data_Path / r"olist_order_items_dataset.csv")
payments = pd.read_csv(Raw_Data_Path / r"olist_order_payments_dataset.csv")
reviews = pd.read_csv(Raw_Data_Path / r"olist_order_reviews_dataset.csv")
orders = pd.read_csv(Raw_Data_Path / r"olist_orders_dataset.csv")
products = pd.read_csv(Raw_Data_Path / r"olist_products_dataset.csv")
sellers = pd.read_csv(Raw_Data_Path / r"olist_sellers_dataset.csv")
category_translation = pd.read_csv(Raw_Data_Path / r"product_category_name_translation.csv")

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

print(f"Total datasets loaded: {len(datasets)}")

Total datasets loaded: 9


In [6]:
for name, df in datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

print("Column names standardized.")

Column names standardized.


In [7]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name:20} -> {duplicate_count:,} duplicate rows")

for name, df in datasets.items():
    before = len(df)
    datasets[name] = df.drop_duplicates().copy()
    after = len(datasets[name])
    print(
        f"{name:20} -> removed {before - after:,} duplicate rows"
    )

customers            -> 0 duplicate rows
orders               -> 0 duplicate rows
order_items          -> 0 duplicate rows
products             -> 0 duplicate rows
sellers              -> 0 duplicate rows
payments             -> 0 duplicate rows
reviews              -> 0 duplicate rows
geolocation          -> 261,831 duplicate rows
category_translation -> 0 duplicate rows
customers            -> removed 0 duplicate rows
orders               -> removed 0 duplicate rows
order_items          -> removed 0 duplicate rows
products             -> removed 0 duplicate rows
sellers              -> removed 0 duplicate rows
payments             -> removed 0 duplicate rows
reviews              -> removed 0 duplicate rows
geolocation          -> removed 261,831 duplicate rows
category_translation -> removed 0 duplicate rows


In [7]:
customers = datasets["customers"]
orders = datasets["orders"]
order_items = datasets["order_items"]
products = datasets["products"]
sellers = datasets["sellers"]
payments = datasets["payments"]
reviews = datasets["reviews"]
geolocation = datasets["geolocation"]
category_translation = datasets["category_translation"]

print("DataFrames updated.")

DataFrames updated.


In [8]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

print(orders[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [9]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

print(order_items["shipping_limit_date"].dtype)

datetime64[us]


In [10]:
print("Invalid purchase dates:",
      orders["order_purchase_timestamp"].isna().sum())

print("Invalid shipping dates:",
      order_items["shipping_limit_date"].isna().sum())

Invalid purchase dates: 0
Invalid shipping dates: 0


In [11]:
text_columns = {
    "customers": [
        "customer_city",
        "customer_state"
    ],
    "orders": [
        "order_status"
    ],
    "products": [
        "product_category_name"
    ],
    "sellers": [
        "seller_city",
        "seller_state"
    ],
    "payments": [
        "payment_type"
    ]
}

for dataset_name, columns in text_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        if column in df.columns:
            df[column] = df[column].str.strip()

In [12]:
categorical_columns = {
    "customers": ["customer_state"],
    "orders": ["order_status"],
    "products": ["product_category_name"],
    "sellers": ["seller_state"],
    "payments": ["payment_type"]
}

for dataset_name, columns in categorical_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        if column in df.columns:
            df[column] = df[column].astype("category")

print("Categorical columns converted.")

Categorical columns converted.


In [13]:
numeric_columns = {
    "order_items": [
        "order_item_id",
        "price",
        "freight_value"
    ],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],
    "reviews": [
        "review_score"
    ],
    "products": [
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "geolocation": [
        "geolocation_lat",
        "geolocation_lng"
    ]
}

for dataset_name, columns in numeric_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

print("Numeric columns converted.")

Numeric columns converted.


In [14]:
print("Negative prices:",
      (order_items["price"] < 0).sum())

print("Negative freight values:",
      (order_items["freight_value"] < 0).sum())

print("Invalid review scores:",
      (~reviews["review_score"].between(1, 5)).sum())

print("Negative payment values:",
      (payments["payment_value"] < 0).sum())

Negative prices: 0
Negative freight values: 0
Invalid review scores: 0
Negative payment values: 0


In [15]:
print(
    "Missing product categories:",
    products["product_category_name"].isna().sum()
)

Missing product categories: 610


In [16]:
products["product_category_name"] = (
    products["product_category_name"]
    .cat.add_categories(["unknown"])
    .fillna("unknown")
)
print(
    "Missing product categories after cleaning:",
    products["product_category_name"].isna().sum()
)

Missing product categories after cleaning: 0


In [17]:
products["product_name_lenght"] = (
    products["product_name_lenght"].fillna(0)
)

products["product_description_lenght"] = (
    products["product_description_lenght"].fillna(0)
)

products["product_photos_qty"] = (
    products["product_photos_qty"].fillna(0)
)

products["product_category_name"] = (
    products["product_category_name"].fillna(0)
)
for col in [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]:
    products[col] = products[col].fillna(products[col].median())

In [18]:
category_translation["product_category_name"] = (
    category_translation["product_category_name"]
    .str.strip()
)

category_translation["product_category_name_english"] = (
    category_translation["product_category_name_english"]
    .str.strip()
)

print("Category translation cleaned.")

Category translation cleaned.


In [19]:
review_text_columns = [
    "review_comment_title",
    "review_comment_message"
]

for column in review_text_columns:
    reviews[column] = reviews[column].fillna("")

print("Review text fields cleaned.")

Review text fields cleaned.


In [20]:
key_columns = {
    "customers": [
        "customer_id",
        "customer_unique_id"
    ],
    "orders": [
        "order_id",
        "customer_id"
    ],
    "order_items": [
        "order_id",
        "product_id",
        "seller_id"
    ],
    "products": [
        "product_id"
    ],
    "sellers": [
        "seller_id"
    ],
    "payments": [
        "order_id"
    ],
    "reviews": [
        "review_id",
        "order_id"
    ]
}

for dataset_name, columns in key_columns.items():
    df = datasets[dataset_name]

    print(f"\n{dataset_name.upper()}")

    for column in columns:
        print(
            f"{column:30} missing: "
            f"{df[column].isna().sum():,}"
        )


CUSTOMERS
customer_id                    missing: 0
customer_unique_id             missing: 0

ORDERS
order_id                       missing: 0
customer_id                    missing: 0

ORDER_ITEMS
order_id                       missing: 0
product_id                     missing: 0
seller_id                      missing: 0

PRODUCTS
product_id                     missing: 0

SELLERS
seller_id                      missing: 0

PAYMENTS
order_id                       missing: 0

REVIEWS
review_id                      missing: 0
order_id                       missing: 0


In [21]:
valid_customer_ids = orders["customer_id"].isin(
    customers["customer_id"]
)

print(
    f"Valid order-customer relationships: "
    f"{valid_customer_ids.mean() * 100:.2f}%"
)

valid_product_ids = order_items["product_id"].isin(
    products["product_id"]
)

print(
    f"Valid order-product relationships: "
    f"{valid_product_ids.mean() * 100:.2f}%"
)

valid_seller_ids = order_items["seller_id"].isin(
    sellers["seller_id"]
)

print(
    f"Valid order-seller relationships: "
    f"{valid_seller_ids.mean() * 100:.2f}%"
)

Valid order-customer relationships: 100.00%
Valid order-product relationships: 100.00%
Valid order-seller relationships: 100.00%


In [22]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

print("Dataset dictionary refreshed.")

Dataset dictionary refreshed.


In [23]:
missing_report = []

for name, df in datasets.items():
    total_missing = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    missing_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": total_missing,
        "missing_percentage": round(
            (total_missing / total_cells) * 100, 4
        )
    })

missing_report_df = pd.DataFrame(missing_report)

missing_report_df

,dataset,rows,columns,missing_cells,missing_percentage
0,customers,99441,5,0,0.0000
1,orders,99441,8,4908,0.6169
2,order_items,112650,7,0,0.0000
3,products,32951,9,0,0.0000
4,sellers,3095,4,0,0.0000
5,payments,103886,5,0,0.0000
6,reviews,99224,7,0,0.0000
7,geolocation,738332,5,0,0.0000
8,category_translation,71,2,0,0.0000


In [24]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing)


orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


In [25]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

orders.isna().sum()
print(products["product_category_name"].isna().sum())
missing_orders = orders.isna().sum()
print(missing_orders[missing_orders > 0])

0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


In [26]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}
print("Dataset dictionary refreshed.")

Dataset dictionary refreshed.


In [27]:
missing_report = []

for name, df in datasets.items():
    total_missing = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    missing_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": total_missing,
        "missing_percentage": round(
            (total_missing / total_cells) * 100, 4
        )
    })

missing_report_df = pd.DataFrame(missing_report)

missing_report_df

,dataset,rows,columns,missing_cells,missing_percentage
0,customers,99441,5,0,0.0000
1,orders,99441,8,4908,0.6169
2,order_items,112650,7,0,0.0000
3,products,32951,9,0,0.0000
4,sellers,3095,4,0,0.0000
5,payments,103886,5,0,0.0000
6,reviews,99224,7,0,0.0000
7,geolocation,738332,5,0,0.0000
8,category_translation,71,2,0,0.0000


In [28]:
print("ORDERS:")
print(orders.isna().sum()[orders.isna().sum() > 0])

print("\nPRODUCTS:")
print(products.isna().sum()[products.isna().sum() > 0])

ORDERS:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

PRODUCTS:
Series([], dtype: int64)


In [29]:
products["product_name_lenght"] = (
    products["product_name_lenght"].fillna(0)
)

products["product_description_lenght"] = (
    products["product_description_lenght"].fillna(0)
)

products["product_photos_qty"] = (
    products["product_photos_qty"].fillna(0)
)

products["product_category_name"] = (
    products["product_category_name"].fillna(0)
)
for col in [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]:
    products[col] = products[col].fillna(products[col].median())

In [30]:
print(products.isna().sum()[products.isna().sum() > 0])

Series([], dtype: int64)


In [31]:
print(
    orders.loc[
        orders["order_delivered_customer_date"].isna(),
        "order_status"
    ].value_counts()
)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [32]:
print(
    orders.loc[
        orders["order_delivered_carrier_date"].isna(),
        "order_status"
    ].value_counts()
)

print(
    orders.loc[
        orders["order_approved_at"].isna(),
        "order_status"
    ].value_counts()
)

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
shipped          0
Name: count, dtype: int64
order_status
canceled       141
delivered       14
created          5
approved         0
invoiced         0
processing       0
shipped          0
unavailable      0
Name: count, dtype: int64


In [33]:
print("PRODUCTS MISSING:")
print(products.isna().sum()[products.isna().sum() > 0])
datasets["products"] = products
datasets["orders"] = orders

PRODUCTS MISSING:
Series([], dtype: int64)


In [34]:
missing_report = []

for name, df in datasets.items():
    total_missing = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    missing_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": total_missing,
        "missing_percentage": round(
            (total_missing / total_cells) * 100, 4
        )
    })

missing_report_df = pd.DataFrame(missing_report)

missing_report_df

,dataset,rows,columns,missing_cells,missing_percentage
0,customers,99441,5,0,0.0000
1,orders,99441,8,4908,0.6169
2,order_items,112650,7,0,0.0000
3,products,32951,9,0,0.0000
4,sellers,3095,4,0,0.0000
5,payments,103886,5,0,0.0000
6,reviews,99224,7,0,0.0000
7,geolocation,738332,5,0,0.0000
8,category_translation,71,2,0,0.0000


In [35]:
print("Missing order_approved_at:")
print(
    orders.loc[
        orders["order_approved_at"].isna(),
        "order_status"
    ].value_counts()
)

print("\nMissing order_delivered_carrier_date:")
print(
    orders.loc[
        orders["order_delivered_carrier_date"].isna(),
        "order_status"
    ].value_counts()
)

print("\nMissing order_delivered_customer_date:")
print(
    orders.loc[
        orders["order_delivered_customer_date"].isna(),
        "order_status"
    ].value_counts()
)

Missing order_approved_at:
order_status
canceled       141
delivered       14
created          5
approved         0
invoiced         0
processing       0
shipped          0
unavailable      0
Name: count, dtype: int64

Missing order_delivered_carrier_date:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
shipped          0
Name: count, dtype: int64

Missing order_delivered_customer_date:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [36]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [37]:
orders.dropna(
    subset=[
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ],
    inplace=True
)

print("Remaining missing values in orders:")
print(orders.isna().sum()[orders.isna().sum() > 0])
print("Orders remaining:", len(orders))

Remaining missing values in orders:
Series([], dtype: int64)
Orders remaining: 96461


In [38]:
datasets["orders"] = orders

In [39]:
missing_report = []

for name, df in datasets.items():
    total_missing = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    missing_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": total_missing,
        "missing_percentage": round(
            (total_missing / total_cells) * 100, 4
        )
    })

missing_report_df = pd.DataFrame(missing_report)

missing_report_df

,dataset,rows,columns,missing_cells,missing_percentage
0,customers,99441,5,0,0.0
1,orders,96461,8,0,0.0
2,order_items,112650,7,0,0.0
3,products,32951,9,0,0.0
4,sellers,3095,4,0,0.0
5,payments,103886,5,0,0.0
6,reviews,99224,7,0,0.0
7,geolocation,738332,5,0,0.0
8,category_translation,71,2,0,0.0


In [40]:
output_files = {
    "customers": "customers_clean.csv",
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "category_translation": "category_translation_clean.csv"
}

for name, file_name in output_files.items():
    datasets[name].to_csv(
        Clean_Data_Path/file_name,
        index=False
    )

print("All clean datasets saved successfully.")

All clean datasets saved successfully.


In [41]:
Data_Path = r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\tables/"
cleaning_report = []

for name, df in datasets.items():
    cleaning_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

cleaning_report_df = pd.DataFrame(cleaning_report)

cleaning_report_df.to_csv(
    Data_Path + r"cleaning_report.csv",
    index=False
)

print("Cleaning report saved successfully.")

Cleaning report saved successfully.
